# 深度学习数学基础：深度分离（Depth Separation）实验研究
**课程项目:** EECSE 6699 - Mathematics of Deep Learning  
**核心目标:** 验证神经网络中“深度”对“表达效率”的指数级贡献。

---

## 1. 理论阐述 (Theoretical Foundation)
根据 **Telgarsky (2016)** 的研究，深层 ReLU 网络在表达具有高频振荡特性的函数（如锯齿函数）时，比浅层网络具有显著的指数级优势。

### 数学原理：ReLU 的复合效应
- 考虑基础镜像算子：$\phi(x) = |2x - 1|$。这是一个简单的 2 层 ReLU 网络可以完美表达的函数。
- 当我们将该算子进行 $L$ 次复合（Composition）时：$f_L(x) = \phi(\phi(...\phi(x)...))$。
- **分段线性特征**: 每复合一次，函数的线性分段数量翻倍。因此，$L$ 层网络可以生成 $2^L$ 个分段。
- **深度分离定理**: 若要用一个只有 2 层的浅层网络达到同样的拟合精度，所需的神经元数量将随 $L$ 指数级增长。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# 设置全局随机种子
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. 定义目标锯齿函数 (Target Function)
我们构造一个具有 $2^4 = 16$ 个波峰的锯齿函数作为拟合目标。

In [ ]:
def sawtooth_target(x, k=4):
    res = x
    for _ in range(k):
        res = torch.abs(2 * res - 1)
    return res

# 生成测试数据用于可视化
x_plot = torch.linspace(0, 1, 1000).view(-1, 1).to(device)
y_plot = sawtooth_target(x_plot, k=4)

plt.figure(figsize=(10, 4))
plt.plot(x_plot.cpu().numpy(), y_plot.cpu().numpy(), 'b-', label='Target $f_{k=4}(x)$')
plt.title("Target Sawtooth Function with 16 Peaks")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 3. 模型构建与参数对等性分析
为了确保实验的严谨性，我们设计的“浅层网络”在**参数总量**上应不低于“深层网络”。

In [ ]:
class ModelBuilder(nn.Module):
    def __init__(self, depth, width):
        super().__init__()
        layers = [nn.Linear(1, width), nn.ReLU()]
        for _ in range(depth - 2):
            layers.extend([nn.Linear(width, width), nn.ReLU()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x): 
        return self.net(x)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 1. 配置深层网络 (Deep & Narrow)
deep_model = ModelBuilder(depth=9, width=4).to(device)
d_params = count_parameters(deep_model)

# 2. 配置参数对等的浅层网络 (Shallow & Wide)
# 参数量计算: (1*W + W) + (W*1 + 1) = 3W + 1 (对于 2 层网络)
s_width = (d_params - 1) // 3
shallow_model = ModelBuilder(depth=2, width=s_width).to(device)

print(f"Deep Model: Depth=9, Width=4, Params={d_params}")
print(f"Shallow Model: Depth=2, Width={s_width}, Params={count_parameters(shallow_model)}")

## 4. 训练实验 (Training Session)
我们将使用相同的超参数对两个模型进行训练。

In [ ]:
def train(model, epochs=15000, lr=0.003):
    x_train = torch.linspace(0, 1, 1200).view(-1, 1).to(device)
    y_train = sawtooth_target(x_train, k=4)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    losses = []
    for i in range(epochs):
        optimizer.zero_grad()
        pred = model(x_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
        if i % 3000 == 0:
            print(f"Step {i}, Loss: {loss.item():.6f}")
            
    return losses

print("Training Deep Model...")
deep_losses = train(deep_model)

print("\nTraining Shallow Model...")
shallow_losses = train(shallow_model)

## 5. 结果分析与可视化 (Analysis & Visualization)

In [ ]:
with torch.no_grad():
    y_deep_pred = deep_model(x_plot).cpu().numpy()
    y_shallow_pred = shallow_model(x_plot).cpu().numpy()
    y_true = y_plot.cpu().numpy()

plt.figure(figsize=(16, 6))

# 子图 1: 拟合效果对比
plt.subplot(1, 2, 1)
plt.plot(x_plot.cpu(), y_true, 'k--', alpha=0.3, label='Ground Truth')
plt.plot(x_plot.cpu(), y_deep_pred, 'g-', linewidth=2, label='Deep (L=9, W=4)')
plt.plot(x_plot.cpu(), y_shallow_pred, 'r-', linewidth=1.5, label=f'Shallow (L=2, W={s_width})')
plt.title("Approximation Results")
plt.legend()
plt.grid(True, alpha=0.2)

# 子图 2: 训练曲线对比
plt.subplot(1, 2, 2)
plt.plot(deep_losses, 'g', label='Deep Net Loss')
plt.plot(shallow_losses, 'r', label='Shallow Net Loss')
plt.yscale('log')
plt.title("Convergence Rate (Log Scale)")
plt.xlabel("Epochs")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 6. 实验洞察与论文讨论点 (Project Discussion)

1. **表达能力的本质**: 浅层网络即便拥有更多的神经元（宽度极大），在拟合高频信号时依然会显得“平滑”。这是因为浅层架构缺乏通过层级复合产生大量线性分段的数学机制。
2. **优化难度**: 你可能会观察到深层网络的训练曲线初期下降较慢，或者更容易陷入死神经元（ReLU Dying）。这引出了关于初始化（Initialization）和梯度流动（Gradient Flow）的进一步讨论。
3. **参数效率**: 实验有力地证明了，在表示特定逻辑/拓扑结构时，增加“层数”比单纯增加“参数总量”要高效得多。